In [ ]:
import sys
sys.path.append("..")

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    DINOv3ViTModel,
    UperNetForSemanticSegmentation, 
    DINOv3ViTConfig,
    UperNetConfig,
    UperNetForSemanticSegmentation,
    ResNetConfig,
    AutoBackbone,
    BeitForSemanticSegmentation,
    BeitConfig,
)
from transformers.models.beit.modeling_beit import BeitFCNHead, BeitFPNNeck, BeitUperHead
from transformers.configuration_utils import PreTrainedConfig, SubConfigSpec
from transformers.backbone_utils import load_backbone
from transformers.modeling_utils import PreTrainedModel
from transformers.modeling_outputs import SemanticSegmenterOutput

from src.model import DINOv3ForSemanticSegmentationConfig, DINOv3ForSemanticSegmentation


# Model Decision Notes:

Dinov3
| size | s | b | l |
|------|---|---|---|
| hidden_size | 384 | 768 | 1024 |
| layers | 12 | 12 | 24 | 



Exploration Notes:
1. Directly initializing dinov3 with upernet does not create feature pyramids
2. We follow Beit architecture and use their FPN neck and upernet head



Parameters Count:
| size  | total   | backbone | fpn_neck + decode_head | fpn_neck | decode_head | auxiliary_head |
|-------|---------|----------|------------------------|----------|-------------|----------------|
| small | 41.23M  | 21.60M   | 18.74M                 | 1.77M    | 16.97M      | 0.89M          |
| base  | 162.38M | 85.66M   | 74.94M                 | 7.08M    | 67.86M      | 1.77M          |
| large | 438.71M | 303.13M  | 133.22M                | 12.59M   | 120.63M     | 2.36M          |


the trainable vs non-trainable parameters dont change linearly with backbone size.
trainable % : s 47.6%, b 47.2%, l 30.9%

In [ ]:
# a = DINOv3ViTModel.from_pretrained("facebook/dinov3-vits16-pretrain-lvd1689m")
# a = DINOv3ViTModel.from_pretrained("facebook/dinov3-vitb16-pretrain-lvd1689m")
# a = DINOv3ViTModel.from_pretrained("facebook/dinov3-vitl16-pretrain-lvd1689m")

In [ ]:
# beit uper head needs pool_scales, hidden_size, num_labels 
# beit fpn neck need hidden_size
# beit fcn head needs hidden_size, auxiliary_channels, auxiliary_num_convs, auxiliary_concat_input

In [ ]:
dinov3_config = DINOv3ViTConfig.from_pretrained(
    "facebook/dinov3-vitl16-pretrain-lvd1689m", 
    out_indices=[4, 6, 8, 12],
    apply_layernorm=True,
)
cfg = DINOv3ForSemanticSegmentationConfig(
    dinov3_config=dinov3_config,
    initializer_range=0.02,
)
model = DINOv3ForSemanticSegmentation(cfg)
model.to("cuda", dtype=torch.bfloat16)


In [ ]:
# freeze dinov3 backbone
for param in model.backbone.parameters():
    param.requires_grad = False


In [ ]:
img = torch.randn(4, 3, 256, 512)
img = img.to("cuda", dtype=torch.bfloat16)
out = model(img)

In [ ]:
out.logits.mean().backward()

In [ ]:
model.backbone.embeddings.patch_embeddings.weight.grad

In [ ]:
model.fpn_neck.fpn2